# Tariff and controller experiments

A reusable local controller with scheduled storage, reserves, persistent staggering,
export caps, and bounded voltage-trend feedback. The controller observes no price.
Each tariff selects a policy using mean household settlement on training weather.

Run the experiment from the repository root:
```sh
.venv/bin/python -m sandbox.experiments --output results/controller_framework
```
This notebook reads the saved run; reopening it does not silently repeat training.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "sandbox").exists() else Path.cwd().parent
RESULTS = ROOT / "results" / "controller_framework"
manifest = json.loads((RESULTS / "manifest.json").read_text())
selection = json.loads((RESULTS / "selection.json").read_text())
metrics = pd.read_csv(RESULTS / "metrics.csv")
tuning = pd.read_csv(RESULTS / "tuning.csv")
households = pd.read_csv(RESULTS / "household_settlements.csv")
print("Weather splits:", manifest["seeds"], "roots:", manifest["roots"])
print("Finalist:", selection)


## Independent test results
Positive settlement means a household receives money. Lower cost per actual load kWh
is better. Official fields named cost_per_kwh use grid imports as their
denominator; these can diverge near zero imports. Use the added cost_per_load_kwh
fields for incidence comparisons. A lower peak alone does not establish a better tariff; inspect
curtailment, synchrony, and incidence alongside it.


In [ ]:
test = metrics.query("split == 'test'")
columns = ["transformer_export_peak_kw", "transformer_draw_peak_kw", "max_ramp_kw",
           "coincidence_factor", "curtailed_share", "community_settlement_chf",
           "tenant_cost_per_load_kwh_chf", "valid_share"]
display(test.groupby("run")[columns].mean().round(4))
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, field, title in zip(axes, columns[:1] + ["curtailed_share", "community_settlement_chf"],
                           ["Export peak (kW)", "Curtailed share", "Community settlement (CHF)"]):
    means = test.groupby("run")[field].mean()
    errors = test.groupby("run")[field].sem()
    ax.bar(range(len(means)), means, yerr=errors, capsize=3, color="#237b80")
    ax.set_xticks(range(len(means)), means.index, rotation=65, ha="right")
    ax.set_title(title)
fig.suptitle("Independent test weather; error bars show standard error")
fig.tight_layout()
fig.savefig(RESULTS / "test_comparison.png", dpi=160, bbox_inches="tight")


## Tariff response and household incidence
The frozen bank is searched identically for each tariff. A gap of zero identifies
the training winner; small CHF gaps indicate uncertain private preferences.
Tenants are included below even though the standard tuner excludes them.


In [ ]:
display(tuning.query("controller == 'family'").sort_values(["tariff", "gap_chf"])
        .groupby("tariff").head(3))
display(households.query("split == 'test'")
        .groupby(["run", "household"]).settlement_chf.mean().unstack().round(3))
paired = json.loads((RESULTS / "paired_deltas.json").read_text())
display(pd.DataFrame(paired).T.round(5))

incidence = households.query("split == 'test'").groupby(["run", "household"])[["settlement_chf", "load_kwh"]].sum()
incidence["cost_per_actual_load_kwh_chf"] = -incidence.settlement_chf / incidence.load_kwh
display(incidence.cost_per_actual_load_kwh_chf.unstack().round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for label in ["fair_leg_default_base", "fair_leg_tuned_family", "tariff_tuned_family"]:
    frame = pd.read_csv(RESULTS / f"{label}_feeder_trace.csv")
    day = frame.query("day == 2").sort_values("hour")
    ax.plot(day.hour, day.transformer_kw, label=label)
ax.axhline(0, color="grey", lw=0.7)
ax.set(xlabel="Hour", ylabel="Transformer kW (+ import / − export)",
       title="One illustrative test day; aggregate conclusions use all test weeks")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS / "feeder_day.png", dpi=160)


## Interpretation limits

The response is a shared-policy approximation, not a market equilibrium.
Use `sandbox.response_audit.audit_deviations` to check unilateral policy changes.
The official `score()` report remains available through `sandbox.my_idea` and
uses its original tuning and seed conventions. This experiment's fair-LEG
family comparator is explicitly tuned and uses separate weather roots.

The bank is deliberately finite. Expand it after diagnosing a missing behaviour,
then rerun every tariff using the same expanded bank. Battery wear is not included
in the current household settlement objective.
